# NB02 — Biome Stratification & Aggregation

**Environment:** BERDL JupyterHub (Spark)

**Purpose:** Join per-cluster coverage (from NB01) to species assignment (`marker_gene_clusters`) → genome environment (`genome_environment.csv`) → biome. Aggregate to a biome × functional-category matrix.

**Inputs:**
- `.../per_cluster_coverage.parquet` (from NB01)
- `plant_microbiome_ecotypes/data/marker_gene_clusters.csv` (588K rows)
- `plant_microbiome_ecotypes/data/genome_environment.csv` (293K rows)

**Outputs:**
- `data/biome_coverage_matrix.csv` — cell-level counts and rates (biome × function × core status)
- `data/species_biome_assignment.csv` — majority-vote biome per species (intermediate)

**Pitfalls baked in:**
- GTDB prefix mismatch: `genome_environment.genome_id` uses `RS_GCF_*`/`GB_GCA_*`; `marker_gene_clusters` uses `gtdb_species_clade_id` (species-level, no prefix issue at this join, but check any tree/pair join downstream).
- `compartment` is plant-centric — many non-plant genomes will have `compartment='other'` or NULL. Use `env_broad_scale` as fallback.
- Many-to-many: one cluster maps to multiple species; one species can have many biome-compartment memberships. Aggregate at cluster-species-biome triple, then roll up.

In [ ]:
from berdl_notebook_utils.setup_spark_session import get_spark_session
from pyspark.sql.functions import col, count, sum as spark_sum, when, row_number
from pyspark.sql.window import Window

spark = get_spark_session()

PER_CLUSTER_PATH = "s3a://cdm-lake/tenant-general-warehouse/microbialdiscoveryforge/projects/structural_coverage_biome/data/per_cluster_coverage.parquet"
MARKER_GC_PATH = "file:///home/aparkin/BERIL-research-observatory/projects/plant_microbiome_ecotypes/data/marker_gene_clusters.csv"
GENOME_ENV_PATH = "file:///home/aparkin/BERIL-research-observatory/projects/plant_microbiome_ecotypes/data/genome_environment.csv"
OUTPUT_DIR = "data"

## Load inputs

In [ ]:
per_cluster = spark.read.parquet(PER_CLUSTER_PATH)
mgc = spark.read.csv(MARKER_GC_PATH, header=True, inferSchema=True).select(
    "gene_cluster_id", "gtdb_species_clade_id", "is_core", "is_auxiliary", "is_singleton"
)
genome_env = spark.read.csv(GENOME_ENV_PATH, header=True, inferSchema=True).select(
    "genome_id", "gtdb_species_clade_id", "compartment", "env_broad_scale", "phylum"
)

## Species → majority biome (compartment × env_broad_scale)

In [ ]:
species_env_counts = genome_env.groupBy(
    "gtdb_species_clade_id", "phylum", "compartment", "env_broad_scale"
).agg(count("*").alias("n_genomes"))

w = Window.partitionBy("gtdb_species_clade_id").orderBy(col("n_genomes").desc())
species_env = (species_env_counts
    .withColumn("rk", row_number().over(w))
    .filter(col("rk") == 1)
    .drop("rk")
)
species_env.coalesce(1).write.mode("overwrite").csv(f"{OUTPUT_DIR}/species_biome_assignment.csv", header=True)

## Full join → cluster × species × biome

In [ ]:
biome_coverage = (per_cluster
    .join(mgc, on="gene_cluster_id", how="inner")
    .join(species_env, on="gtdb_species_clade_id", how="inner")
)

## Aggregate: biome × KEGG-KO × is_core

In [ ]:
agg = biome_coverage.groupBy(
    "compartment", "env_broad_scale", "kegg_orthology_id", "is_core"
).agg(
    count("*").alias("n_clusters"),
    spark_sum(when(col("pdb_tier") == "direct", 1).otherwise(0)).alias("n_pdb_direct"),
    spark_sum(when(col("pdb_tier") == "homolog", 1).otherwise(0)).alias("n_pdb_homolog"),
    spark_sum(when(col("pdb_tier") == "none", 1).otherwise(0)).alias("n_pdb_none"),
    spark_sum(when(col("af_tier") == "confident", 1).otherwise(0)).alias("n_af_confident"),
    spark_sum(when(col("af_tier") == "low_confidence", 1).otherwise(0)).alias("n_af_low"),
    spark_sum(when(col("af_tier") == "none", 1).otherwise(0)).alias("n_af_none"),
)

agg.coalesce(1).write.mode("overwrite").csv(f"{OUTPUT_DIR}/biome_coverage_matrix.csv", header=True)

## Also emit a per-cluster-with-biome file for NB04 priority list

In [ ]:
biome_coverage.write.mode("overwrite").parquet(
    "s3a://cdm-lake/tenant-general-warehouse/microbialdiscoveryforge/projects/structural_coverage_biome/data/cluster_biome_coverage.parquet"
)